# 08 Full Run

**Purpose:** Run the full end-to-end OCT barcoding pipeline from exported E2E scans to barcode predictions and measurements.

**Inputs:**
- `data/heyex/`: raw exported E2E files organized by patient folder.
- `data/processed/labels/clinician/barcode_labels.csv`: clinician labels once available.

**Main task:**
Provide a single reproducible pipeline run that executes preprocessing, barcode classification, and barcode measurement in order.

**Planned workflow:**
1. Run E2E preprocessing from raw data.
2. Save ROI volumes and preprocessing QC.
3. Train or load the ResNet barcode classifier.
4. Generate barcode predictions.
5. Run barcode measurements only on positive scans/B-scans.
6. Save all tables, models, figures, and summary files.
7. Print final pipeline status.

**Code organization:**
- This notebook should contain almost no low-level code.
- It should call:
  - `src/barcode/preprocessing.py`
  - `src/barcode/resnet.py`
  - `src/barcode/measurement.py`
  - `src/barcode/validation.py` if needed.
- Any reusable orchestration function can later go in `src/barcode/pipeline.py`.

**Expected outputs:**
- `data/processed/qc/preprocessing_qc.csv`
- `data/processed/labels/barcode_label_template.csv`
- `data/processed/models/`
- `data/processed/predictions/barcode_predictions.csv`
- `data/processed/features/barcode_bscan_measurements.csv`
- `data/processed/features/barcode_volume_measurements.csv`
- `data/processed/qc/full_run_summary.json`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "heyex"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

LABEL_FILE = PROCESSED_DIR / "labels" / "clinician" / "barcode_labels.csv"
C8_CHECKPOINT = PROJECT_ROOT / "results" / "models" / "resnet50_oct_c8_layer4_finetuned.pt"

In [ ]:
from barcode.preprocessing import run_preprocessing_pipeline
from barcode.resnet import run_resnet_finetuning
from barcode.measurement import run_measurement_pipeline

In [ ]:
qc_df, label_df = run_preprocessing_pipeline(
    data_dir=RAW_DATA_DIR,
    processed_dir=PROCESSED_DIR,
    n_patients=None,
    save_flattened=False,
)

display(qc_df.head())
display(label_df.head())

In [ ]:
if LABEL_FILE.exists():
    history_df, volume_pred_df = run_resnet_finetuning(
        processed_dir=PROCESSED_DIR,
        label_file=LABEL_FILE,
        c8_checkpoint_path=C8_CHECKPOINT,
        label_col="barcode_volume_status",
        use_positive_range=True,
        batch_size=16,
        epochs=5,
        lr=1e-4,
        weight_decay=1e-4,
        unfreeze_final_block=True,
    )

    display(history_df)
    display(volume_pred_df.head())
else:
    print("Label file not found yet. Skipping ResNet fine-tuning.")
    print("Expected:", LABEL_FILE)

In [ ]:
PREDICTIONS_FILE = PROCESSED_DIR / "predictions" / "barcode_volume_predictions.csv"

if PREDICTIONS_FILE.exists():
    bscan_measure_df, volume_measure_df = run_measurement_pipeline(
        processed_dir=PROCESSED_DIR,
        predictions_file=PREDICTIONS_FILE,
    )

    display(bscan_measure_df.head())
    display(volume_measure_df.head())
else:
    print("Prediction file not found yet. Skipping barcode measurements.")
    print("Expected:", PREDICTIONS_FILE)

In [ ]:
for path in [
    PROCESSED_DIR / "qc" / "preprocessing_qc.csv",
    PROCESSED_DIR / "labels" / "barcode_label_template.csv",
    PROCESSED_DIR / "models" / "barcode_resnet.pt",
    PROCESSED_DIR / "predictions" / "barcode_slice_predictions.csv",
    PROCESSED_DIR / "predictions" / "barcode_volume_predictions.csv",
    PROCESSED_DIR / "features" / "barcode_bscan_measurements.csv",
    PROCESSED_DIR / "features" / "barcode_volume_measurements.csv",
]:
    print(path, "exists:", path.exists())